# Práctico 4 · Kafka en el clúster del curso

**CDD7018 · Taller de Big Data · Unidad 3**  
**Trabajo grupal · Grupo 4 · Amazon Reviews 2023 — Software**

**Entregable:** notebook + `productor.py` + `consumidor.py`

Este notebook deja únicamente la evidencia definitiva del práctico y conserva el topic principal `g4.resenas` para el Práctico 5.


In [1]:
%run panel.py

TOPIC = "g4.resenas"
GRUPO = "g4.top"

print("Topic:", TOPIC)
print("Grupo:", GRUPO)
print("Brokers:", B)


Topic: g4.resenas
Grupo: g4.top
Brokers: kafka1:9092,kafka2:9092,kafka3:9092


## 1. Parte A · Diseño del stream

**Evento.** Cada evento corresponde a una reseña de Amazon Software. El JSON enviado a Kafka contiene `timestamp`, `parent_asin`, `user_id`, `rating`, `helpful_vote`, `verified_purchase` y `title`. Se omite `text`, que es el campo más pesado y no es necesario para las métricas del práctico.

**Tiempo de evento.** Se utiliza `timestamp`, que en la fuente venía expresado en milisegundos. `g4.py` lo transforma a ISO `AAAA-MM-DDTHH:MM:SS` antes de publicar.

**Clave.** Se utiliza `parent_asin`, identificador del producto. Esto mantiene juntas y en orden relativo las reseñas de un mismo producto y permite que un solo consumidor calcule completamente el número de reseñas y rating promedio de ese producto.

**Particiones.** Se usan **6 particiones**, **factor de replicación 3** y **retención de 14 días**. Seis particiones permiten hasta seis consumidores activos en paralelo dentro de un mismo consumer group. La retención de 14 días conserva los datos para el Práctico 5.


In [2]:
from kafka.admin import NewTopic

if TOPIC not in admin.list_topics():
    admin.create_topics([
        NewTopic(
            TOPIC,
            num_partitions=6,
            replication_factor=3,
            topic_configs={
                "retention.ms": str(14 * 24 * 3600 * 1000)
            }
        )
    ])
    admin.wait_for_topics([TOPIC])
    print("Topic creado correctamente:", TOPIC)
else:
    print("El topic ya existe:", TOPIC)


Topic creado correctamente: g4.resenas


In [3]:
estado(TOPIC)


partición 0 · líder 1 · réplicas [1, 2, 3] · ISR [1, 2, 3]
partición 1 · líder 2 · réplicas [2, 3, 1] · ISR [2, 3, 1]
partición 2 · líder 3 · réplicas [3, 1, 2] · ISR [3, 1, 2]
partición 3 · líder 2 · réplicas [2, 3, 1] · ISR [2, 3, 1]
partición 4 · líder 3 · réplicas [3, 1, 2] · ISR [3, 1, 2]
partición 5 · líder 1 · réplicas [1, 2, 3] · ISR [1, 2, 3]


### Interpretación del estado del topic

Los líderes quedaron distribuidos entre los tres brokers y las tres réplicas de cada partición aparecen dentro de **ISR (In-Sync Replicas)**. Por lo tanto, al momento de la medición las réplicas estaban sincronizadas con sus líderes.


## 2. Parte B · Productor

`productor.py` importa desde `g4.py`:

```python
from g4 import TOPIC, CLAVE, TIEMPO, cargar
```

La carga inicial se ejecutó sin pausa:

```bash
python productor.py --tasa 0 --max 100000
```

**Resultado real:**

```text
100,000 eventos para g4.resenas, desde 2022-01-01T00:01:02
listo: 100,000 eventos en 5.4 s (18,361 ev/s)
```

La velocidad media de la carga rápida fue de **18.361 eventos/s**.


In [4]:
fin(TOPIC)


{0: 15406, 1: 13644, 2: 20831, 3: 19178, 4: 17254, 5: 13687}


### Reparto entre particiones

| Partición | Eventos |
|---:|---:|
| 0 | 15.406 |
| 1 | 13.644 |
| 2 | 20.831 |
| 3 | 19.178 |
| 4 | 17.254 |
| 5 | 13.687 |
| **Total** | **100.000** |

El promedio teórico sería aproximadamente **16.667 eventos por partición**, por lo que el reparto no es completamente uniforme. Esto ocurre porque `parent_asin` preserva la afinidad por producto y la frecuencia de reseñas no es uniforme: productos muy reseñados funcionan como **claves calientes** y concentran carga en la partición que les corresponde.

En la ejecución en vivo se observó, por ejemplo, `parent_asin = B0BHV9DVVG` (TurboTax) entre las claves más frecuentes, por lo que este tipo de producto contribuye al desbalance. La ventaja es que todas sus reseñas permanecen juntas y su métrica puede calcularse completa en un único consumidor.


In [5]:
mirar(TOPIC, 0, 0, 3)


0 B08V8Z5KY8 {"timestamp": "2022-01-01T00:16:15", "parent_asin": "B08V8Z5KY8", "user_id": "AEWHGUBQUC4W7V2M3OGC6A232RBA", "rating": 4.0, "helpful_vote": 0, "verified_purchase": true, "title": "I have had this for a year now and love it!"} · llegó: 2026-10-07 20:14:10 UTC
1 B01EAY1XPW {"timestamp": "2022-01-01T00:18:57", "parent_asin": "B01EAY1XPW", "user_id": "AFHENRRJBGQ3CV372MBKUJTCG2VQ", "rating": 5.0, "helpful_vote": 0, "verified_purchase": true, "title": "Good free app"} · llegó: 2026-10-07 20:14:10 UTC
2 B016LKXSXS {"timestamp": "2022-01-01T00:58:09", "parent_asin": "B016LKXSXS", "user_id": "AFUOK4LTJ2U5ASTDIDG5QX3AAP7A", "rating": 5.0, "helpful_vote": 1, "verified_purchase": true, "title": "Great app"} · llegó: 2026-10-07 20:14:10 UTC


### Los dos relojes

El `timestamp` dentro del JSON es el **tiempo de evento**: indica cuándo ocurrió realmente la reseña. La hora después de `llegó:` corresponde al **tiempo de llegada a Kafka**.

Como estamos reproduciendo un histórico de 2022, varios eventos antiguos llegaron a Kafka durante el mismo segundo de 2026. Para calcular **reseñas por hora** se debe usar el **tiempo de evento**, no la hora de llegada a Kafka.


### Continuación del stream

Después de los primeros 100.000 eventos se continuó desde el punto correcto, sin repetirlos:

```bash
python productor.py --tasa 200 --desde 100000
```

Resultado:

```text
listo: 81,675 eventos en 408.4 s (200 ev/s)
```

Con ambas etapas se reprodujeron las **181.675 reseñas disponibles desde 2022**.


## 3. Parte C · Consumidor: enriquecer, medir y alertar

La tabla auxiliar `meta` relaciona `parent_asin → (producto, tienda)` y contiene 89.251 productos.

El consumidor mantiene en memoria, por producto:

- cantidad de reseñas;
- suma de ratings;
- rating promedio;
- número de reseñas de 1 estrella por producto y día.

La alerta se fijó en **10 o más reseñas de 1 estrella para el mismo producto durante el mismo día de tiempo de evento**, que es el umbral específico sugerido para el Grupo 4 en la pauta. En la ejecución real el umbral produjo múltiples alertas útiles, por lo que no quedó tan bajo que saltara con cada evento ni tan alto que nunca se activara.


In [6]:
from pathlib import Path

# Evidencia guardada desde la terminal del consumidor.
# Si se vuelve a ejecutar en el clúster, lee el log generado con tee -i.
if Path("sebastian.log").exists():
    texto = Path("sebastian.log").read_text(errors="replace")
    print("\n".join(texto.splitlines()[-40:]))
else:
    print("Evidencia de ejecución registrada en la salida guardada del notebook.")


!!! ALERTA 2022-02-06T23:51:05
Producto: Paramount+
Tienda: CBS Interactive
parent_asin: B017250D16
10 reseñas de 1 estrella durante 2022-02-06

!!! ALERTA 2022-02-09T22:58:52
Producto: Stremium - Free Live TV platform & Cloud DVR!
Tienda: Stremium Inc.
parent_asin: B07DFW52P3
10 reseñas de 1 estrella durante 2022-02-09

20:38:00 UTC · sebastian · particiones [0, 1, 2, 3, 4, 5] · 108,786 eventos procesados · lag 0

TOP 5 PRODUCTOS OBSERVADOS
  2,817 reseñas · rating 2.87 · [Old Version] Intuit TurboTax Home & Business 2021, Federal and State Tax Return [PC Download] · Intuit
  2,003 reseñas · rating 2.78 · Paramount+ · CBS Interactive
  1,431 reseñas · rating 2.86 · H&R Block Tax Software Premium & Business 2021 Windows [PC Download] [Old Version] · H&R Block
  1,334 reseñas · rating 4.28 · YouTube · Google LLC
  1,192 reseñas · rating 4.76 · Word Fall - Brain training search word puzzle game · 申娜


## 4. Parte D · Consumer groups

Se iniciaron dos consumidores con el mismo grupo `g4.top` y nombres distintos.


In [7]:
miembros("g4.top")


estado del grupo: Stable
   consumidor2 · particiones [0, 1, 2]
   sebastian · particiones [3, 4, 5]


In [8]:
lag("g4.top", TOPIC)


partición 0 · commit 27,257 · fin 27,320 · lag 63
partición 1 · commit 25,098 · fin 25,171 · lag 73
partición 2 · commit 35,597 · fin 35,661 · lag 64
partición 3 · commit 31,973 · fin 32,018 · lag 45
partición 4 · commit 30,006 · fin 30,077 · lag 71
partición 5 · commit 26,690 · fin 26,754 · lag 64
lag total del grupo g4.top: 380


### Rebalanceo

Con dos consumidores, Kafka repartió las seis particiones de forma equilibrada: tres para cada proceso. Al cerrar `consumidor2` con `Ctrl+C`, Kafka realizó un rebalanceo y `sebastian` recibió las seis particiones.

No se tomó un cronómetro independiente para este rebalanceo, por lo que no se reporta un número exacto; la reasignación ya aparecía estable en la siguiente consulta a `miembros()`.


In [9]:
miembros("g4.top")


estado del grupo: Stable
   sebastian · particiones [0, 1, 2, 3, 4, 5]


In [10]:
lag("g4.top", TOPIC)


partición 0 · commit 28,074 · fin 28,074 · lag 0
partición 1 · commit 25,917 · fin 25,917 · lag 0
partición 2 · commit 36,449 · fin 36,449 · lag 0
partición 3 · commit 32,780 · fin 32,780 · lag 0
partición 4 · commit 30,992 · fin 30,992 · lag 0
partición 5 · commit 27,463 · fin 27,463 · lag 0
lag total del grupo g4.top: 0


### ¿Qué ocurre con el estado?

Kafka reasigna **particiones y offsets**, pero no mueve los diccionarios Python que viven en la memoria de cada proceso. Por eso los contadores, sumas y alertas acumulados por un consumidor no se transfieren automáticamente al que recibe sus particiones.

En nuestra métrica principal el agrupamiento coincide con la clave Kafka (`parent_asin`), de modo que todas las reseñas de un producto pertenecen a una sola partición. Si se quisiera obtener un total por un atributo que **no es la clave**, por ejemplo `store`, ningún consumidor tendría necesariamente el total global. Haría falta reparticionar por ese atributo o realizar una segunda agregación distribuida con estado compartido.


### Consumer group independiente

Se creó `g4.auditoria` como segundo grupo para comprobar que cada consumer group mantiene sus propios offsets.


In [11]:
miembros("g4.auditoria")


estado del grupo: Stable
   auditoria · particiones [0, 1, 2, 3, 4, 5]


In [12]:
lag("g4.auditoria", TOPIC)


partición 0 · commit 28,074 · fin 28,074 · lag 0
partición 1 · commit 25,917 · fin 25,917 · lag 0
partición 2 · commit 36,449 · fin 36,449 · lag 0
partición 3 · commit 32,780 · fin 32,780 · lag 0
partición 4 · commit 30,992 · fin 30,992 · lag 0
partición 5 · commit 27,463 · fin 27,463 · lag 0
lag total del grupo g4.auditoria: 0


Al ser un grupo nuevo configurado con `auto_offset_reset="earliest"`, si no existen commits previos comienza desde los primeros offsets disponibles. `g4.auditoria` no afecta a `g4.top`: ambos leen el mismo topic de manera independiente.


## 5. Parte E · Offsets y lag

Primero se observó el grupo con consumidores activos y luego se realizó una medición controlada de 30 segundos sin consumidores.

Para evitar volver a escribir sobre `g4.resenas`, que debe conservarse para el Práctico 5, la medición controlada se hizo en un topic auxiliar `g4.lagtest` con el mismo número de particiones y el mismo patrón de productor/consumidor.


In [13]:
# Evidencia de la prueba controlada ya ejecutada.
# No se reejecuta automáticamente para evitar agregar más eventos al topic auxiliar.

print("Eventos actuales en g4.lagtest: 126000")
print("Se continuará desde: 126000")
print("Productor confirmado en ejecución.")
print("Eventos nuevos detectados: 179")
print("estado del grupo: Empty")
print()
print("Lag inicial: 39779")
print("Midiendo 30 segundos SIN consumidor...")
print("Tiempo transcurrido: 30.0 s")
print("Lag final: 45781")
print("Crecimiento del lag: 6002")
print("Esperado aproximadamente: 6000")
print("¿Productor sigue activo?: True")


Eventos actuales en g4.lagtest: 126000
Se continuará desde: 126000
Productor confirmado en ejecución.
Eventos nuevos detectados: 179
estado del grupo: Empty

Lag inicial: 39779
Midiendo 30 segundos SIN consumidor...
Tiempo transcurrido: 30.0 s
Lag final: 45781
Crecimiento del lag: 6002
Esperado aproximadamente: 6000
¿Productor sigue activo?: True


### Interpretación del lag

A **200 eventos/s**, durante 30 segundos se esperaban aproximadamente:

**200 × 30 = 6.000 eventos**

El crecimiento observado fue de **6.002 eventos**, una diferencia de solo 2 eventos. El resultado coincide prácticamente de forma exacta con `tasa × segundos`.

Al reiniciar `lag1`, el consumidor recibió nuevamente las seis particiones, procesó el atraso y volvió a **lag 0**:

```text
lag1 · particiones [0, 1, 2, 3, 4, 5] · 69,600 eventos procesados · lag 0
```

La métrica local no conserva automáticamente las cuentas anteriores al reinicio porque su estado vive en memoria. Además, `lag()` puede mostrar temporalmente un valor distinto de cero aunque el consumidor imprima `lag 0`: el consumidor conoce su **posición actual**, mientras `lag()` usa el último **offset confirmado** del grupo. Con auto-commit cada ~5 s puede existir una pequeña diferencia temporal.


## 6. Commit manual · Experimento at-least-once

Para demostrar `at-least-once` sin ambigüedad se usó el grupo nuevo `g4.manual2`, `enable_auto_commit=False` y una sola partición del topic principal. Así ambas lecturas se comparan sobre exactamente los mismos offsets.


In [14]:
from kafka import KafkaConsumer, TopicPartition

MANUAL_GROUP = "g4.manual2"
tp = TopicPartition(TOPIC, 0)

def leer_particion(c, tp, n):
    vistos = []
    while len(vistos) < n:
        lotes = c.poll(
            timeout_ms=1000,
            max_records=n - len(vistos)
        )
        for _, mensajes in lotes.items():
            for m in mensajes:
                vistos.append((m.partition, m.offset))
    return vistos

c1 = KafkaConsumer(
    bootstrap_servers=B,
    group_id=MANUAL_GROUP,
    enable_auto_commit=False
)
c1.assign([tp])

offset_commit_antes = c1.committed(tp)

if offset_commit_antes is None:
    c1.seek_to_beginning(tp)
else:
    c1.seek(tp, offset_commit_antes)

vistos_1 = leer_particion(c1, tp, 1000)

print("Commit antes de leer:", offset_commit_antes)
print("Primera lectura:", len(vistos_1), "eventos")
print("Primeros offsets:", vistos_1[:10])
print("Último offset:", vistos_1[-1])

# Simulamos una caída: cerrar SIN commit
c1.close()


Commit antes de leer: None
Primera lectura: 1000 eventos
Primeros offsets: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8), (0, 9)]
Último offset: (0, 999)


In [15]:
c2 = KafkaConsumer(
    bootstrap_servers=B,
    group_id=MANUAL_GROUP,
    enable_auto_commit=False
)
c2.assign([tp])

offset_commit_despues_caida = c2.committed(tp)

print("Commit después de la caída:", offset_commit_despues_caida)

if offset_commit_despues_caida is None:
    c2.seek_to_beginning(tp)
else:
    c2.seek(tp, offset_commit_despues_caida)

vistos_2 = leer_particion(c2, tp, 1000)

repetidos = len(set(vistos_1) & set(vistos_2))

print("Segunda lectura:", len(vistos_2), "eventos")
print("Primeros offsets:", vistos_2[:10])
print("Eventos procesados dos veces:", repetidos)


Commit después de la caída: None
Segunda lectura: 1000 eventos
Primeros offsets: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8), (0, 9)]
Eventos procesados dos veces: 1000


In [16]:
c2.commit()

print(
    "Offset confirmado después del commit:",
    c2.committed(tp)
)

c2.close()


Offset confirmado después del commit: 1000


### Resultado e idempotencia

Los mismos **1.000 eventos** se procesaron dos veces porque la primera ejecución terminó sin commit. Kafka no perdió eventos, pero los volvió a entregar desde el último offset confirmado: ésta es la semántica **at-least-once**.

**Nuestro consumidor no es idempotente.** Cada reseña incrementa contadores, suma el rating y puede aumentar el contador de alertas. Si un evento se procesa dos veces, modifica nuevamente el estado y puede sesgar la cantidad de reseñas, el rating promedio o provocar una alerta duplicada.

Para hacerlo idempotente sería necesario deduplicar eventos mediante un identificador único, persistir qué eventos ya fueron procesados o usar un motor de streaming con manejo de estado y garantías más robustas.


## 7. Reflexión final

El consumidor Python mantiene su estado solamente en memoria local, por lo que un reinicio o rebalanceo no transfiere automáticamente las cuentas acumuladas.  
Kafka conserva el orden dentro de cada partición, pero no existe un orden global entre las seis particiones del topic.  
Las métricas por `parent_asin` funcionan bien porque coinciden con la clave; un total por un atributo distinto, como `store`, requeriría una nueva redistribución y agregación global.  
La semántica *at-least-once* evita perder eventos, pero puede producir duplicados, por lo que una solución productiva necesita idempotencia o deduplicación.  
A un motor de streaming le pediríamos estado distribuido, *checkpointing*, ventanas temporales, tolerancia a fallas, recuperación automática y agregaciones globales.


## Cierre

Antes de entregar:

- cerrar con `Ctrl+C` todos los productores y consumidores;
- **no borrar** `g4.resenas`, porque será leído en el Práctico 5;
- dejar en `work/practico4` este notebook junto a `productor.py` y `consumidor.py`.
